## Clinical Correlation — Subcortical Volume (Prodromal Subgroups)

Tests whether per-subject colocalization scores correlate with MoCA, UPDRS-III, and GDS
for the **RBD vs HC** and **Hyposmia vs HC** contrasts.

Covariate adjustment: age, SEX, eTIV, Field Strength.
FDR correction per contrast (Benjamini–Hochberg, 14 maps × 3 clinical vars = 42 tests/contrast).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr, zscore
from statsmodels.stats.multitest import multipletests

In [ ]:
DATA_DIR     = Path("../../data")
RESULTS_PATH = Path("../../results")

df = pd.read_csv(DATA_DIR / "merged_df_volume_subcortical_subgroups.csv", low_memory=False)

# Join eTIV and Field Strength from df1.csv (both needed as covariates)
df1 = pd.read_csv(DATA_DIR / "df1.csv", low_memory=False)
fs_map   = df1[["PATNO", "Field Strength"]].drop_duplicates("PATNO").set_index("PATNO")["Field Strength"]
etiv_map = df1[["PATNO", "eTIV"]].drop_duplicates("PATNO").set_index("PATNO")["eTIV"]
df["Field Strength"] = df["PATNO"].map(fs_map)
df["eTIV"]           = df["PATNO"].map(etiv_map)

print(f"Dataset shape: {df.shape}")
print(f"Contrasts: {df['contrast'].unique()}")
print(f"Field Strength missing: {df['Field Strength'].isna().sum()}")
print(f"eTIV missing: {df['eTIV'].isna().sum()}")
df.head(3)

In [ ]:
clinical_vars = ["moca", "gds", "updrs3_score"]
covariates    = ["age", "SEX", "eTIV", "Field Strength"]

# Clean map names
def clean_map_name(s):
    try:
        system = s.split("|")[0].strip()
        target = s.split("target-")[1].split("_")[0]
        return f"{system} | {target}"
    except Exception:
        return s

df["map_clean"] = df["map"].apply(clean_map_name)
print("Unique maps:", sorted(df["map_clean"].unique()))

In [ ]:
# Residualize clinical variables on covariates
def residualize(df, y, covariates):
    sub = df[[y] + covariates].dropna()
    X   = sm.add_constant(sub[covariates])
    model = sm.OLS(sub[y], X).fit()
    resid = pd.Series(np.nan, index=df.index)
    resid.loc[sub.index] = model.resid.values
    return resid

for var in clinical_vars:
    df[f"{var}_adj"] = residualize(df, var, covariates)

# Z-score residuals
for var in clinical_vars:
    df[f"{var}_z"] = zscore(df[f"{var}_adj"], nan_policy="omit")

print("Residualized and z-scored clinical variables added.")

In [ ]:
# Spearman and Pearson correlations
clinical_z_vars = [f"{v}_z" for v in clinical_vars]
results = []

for contrast in df["contrast"].unique():
    df_c = df[df["contrast"] == contrast]
    for map_name in df_c["map_clean"].unique():
        sub = df_c[df_c["map_clean"] == map_name]
        for var in clinical_z_vars:
            x = sub["colocalization"]
            y = sub[var]
            valid = ~(x.isna() | y.isna())
            if valid.sum() > 2:
                r_p, p_p = pearsonr(x[valid], y[valid])
                r_s, p_s = spearmanr(x[valid], y[valid])
            else:
                r_p = p_p = r_s = p_s = np.nan
            results.append({
                "contrast": contrast, "map": map_name, "clinical_var": var,
                "pearson_r": r_p, "pearson_p": p_p,
                "spearman_r": r_s, "spearman_p": p_s, "n": valid.sum(),
            })

results_df = pd.DataFrame(results)
print(f"Total tests: {len(results_df)}")
results_df.head()

In [ ]:
# FDR correction within each contrast
results_df["pearson_p_fdr"]  = np.nan
results_df["spearman_p_fdr"] = np.nan

for contrast in results_df["contrast"].unique():
    mask_p = (results_df["contrast"] == contrast) & results_df["pearson_p"].notna()
    results_df.loc[mask_p, "pearson_p_fdr"] = multipletests(results_df.loc[mask_p, "pearson_p"], method="fdr_bh")[1]
    mask_s = (results_df["contrast"] == contrast) & results_df["spearman_p"].notna()
    results_df.loc[mask_s, "spearman_p_fdr"] = multipletests(results_df.loc[mask_s, "spearman_p"], method="fdr_bh")[1]

print(f"FDR-significant (Spearman q < 0.05): {(results_df['spearman_p_fdr'] < 0.05).sum()}")
print("\nSignificant results:")
display(results_df[results_df["spearman_p_fdr"] < 0.05].sort_values("spearman_p_fdr"))

In [ ]:
# Save results
out_path = RESULTS_PATH / "clinical_correlation_volume_subcortical_subgroups.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")